# Prompt Engineering 完整课程

本 Notebook 包含 Prompt Engineering 的所有核心技术和实践案例。

In [59]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import LLMConfig
from utils.llm_client import LLMClient
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import Counter
import json
import re
import os

# 从项目根目录的 .env 文件加载 API Key。显式指定路径比 load_dotenv() 自动查找更稳定。
from dotenv import load_dotenv
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

QWEN_API_KEY = os.environ.get("QWEN_API_KEY")
QWEN_BASE_URL = os.environ.get("QWEN_BASE_URL")

if not QWEN_API_KEY or not QWEN_BASE_URL:
    print("⚠️  警告：未找到 QWEN_API_KEY 或 QWEN_BASE_URL，请创建 .env 文件并设置")
    print("参考 .env.example 文件格式")
else:
    print("✅ ClawClaw API Key 和 Base URL 加载成功")
    print("   模型：ClawClaw")
    print(f"   API 地址：{QWEN_BASE_URL}")

# 初始化客户端：max_tokens 要覆盖模型的 reasoning + 最终正文预算。
config = LLMConfig(provider="qwen", model="ClawClaw", temperature=0.3, max_tokens=8192)
client = LLMClient(config)

print("环境配置完成！")

✅ ClawClaw API Key 和 Base URL 加载成功
   模型：ClawClaw
   API 地址：http://122.224.109.54:20001/spiritx-api/v1
环境配置完成！


## 课程目录

1. [基础对话](#基础对话) - 最简单的 LLM 调用
2. [角色设定](#角色设定) - System Prompt 技巧
3. [Few-Shot 学习](#few-shot-学习) - 示例引导
4. [思维链 CoT](#思维链-cot) - 推理过程展示
5. [ReAct 模式](#react-模式) - 推理与行动
6. [高级技巧](#高级技巧) - Self-Consistency, ToT, Chaining
7. [RAG 系统](#rag-系统) - 检索增强生成
8. [教学案例](#教学案例) - 实际应用

---

# 第一章：基础对话

## 1.1 最简单的对话

In [36]:
# 最简单的对话
messages = [{"role": "user", "content": "你好，请介绍一下 Prompt Engineering"}]
response = client.chat(messages)
print(response)



你好！很高兴为你介绍 **Prompt Engineering（提示词工程）**。

简单来说，**Prompt Engineering 是一门通过设计和优化输入给 AI 的指令（Prompt），从而引导大语言模型（LLM）生成高质量、准确且符合预期输出的技术。**

你可以把它想象成**“与超级智能但字面理解力很强的助手沟通的艺术”**。

以下我将从定义、重要性、核心技巧、实战案例以及未来趋势五个方面为你详细介绍。

---

### 1. 什么是 Prompt Engineering？

*   **背景：** 大语言模型（如我、GPT-4、Claude 等）是基于海量数据训练出来的，它们本身没有“意识”，而是根据概率预测下一个字。
*   **核心：** 模型输出的质量高度依赖于你给它的输入（Prompt）。
*   **目标：** 用最少的指令，获得最精准、最有用、最安全的回答。

### 2. 为什么它很重要？

*   **提升效率：** 好的提示词可以让 AI 一次性完成任务，减少反复修改的时间。
*   **控制质量：** 避免 AI 产生“幻觉”（胡说八道）或输出泛泛而谈的内容。
*   **解锁能力：** 通过特定的技巧，可以让 AI 完成复杂的逻辑推理、代码编写、创意写作等任务。
*   **降低成本：** 在商业应用中，更精准的 Prompt 意味着更少的 Token 消耗和更少的错误处理成本。

### 3. 核心技巧与原则（万能公式）

虽然不同的模型有不同的偏好，但以下原则是通用的。一个优秀的 Prompt 通常包含以下要素：

1.  **角色设定 (Role)：** 告诉 AI 它是谁。
    *   *例：* “你是一位资深的 Python 工程师..."
2.  **背景/上下文 (Context)：** 提供任务相关的背景信息。
    *   *例：* “我正在为一个电商网站开发后端接口..."
3.  **明确任务 (Task)：** 具体要做什么，使用动词。
    *   *例：* “请编写一个用户登录的 API 函数..."
4.  **约束条件 (Constraints)：** 限制输出格式、长度、风格等。
    *   *例：* “代码需要包含注释，使用异步处理，输出不超过 50 行。”
5.  **示

## 1.2 多轮对话

In [37]:
# 多轮对话
messages = [
    {"role": "user", "content": "什么是 Chain of Thought？"},
    {"role": "assistant", "content": "Chain of Thought 是一种提示技术，引导模型展示推理过程。"},
    {"role": "user", "content": "它能提高什么类型的任务准确性？"}
]
response = client.chat(messages)
print(response)



Chain of Thought (CoT) 主要能显著提高以下几类需要**复杂推理**、**多步骤处理**或**逻辑推导**的任务的准确性：

1.  **数学与计算问题**
    *   **例子**：算术应用题、代数方程、几何证明。
    *   **原因**：CoT 迫使模型将复杂的计算分解为中间步骤，减少直接跳跃到答案时产生的计算错误。

2.  **逻辑推理任务**
    *   **例子**：逻辑谜题、演绎推理、真假话判断、类比推理。
    *   **原因**：通过展示推理链条，模型能更好地遵循逻辑规则，避免直觉性错误。

3.  **常识推理**
    *   **例子**：理解因果关系、预测行为后果、情境判断。
    *   **原因**：模型需要调用背景知识并连接多个概念，CoT 有助于显式地激活这些知识关联。

4.  **多步骤决策与规划**
    *   **例子**：制定计划、代码调试、复杂问题拆解。
    *   **原因**：将大问题分解为子任务，确保每一步都正确后再进行下一步，降低整体失败率。

5.  **科学解释与推导**
    *   **例子**：解释物理现象、分析实验数据、推导科学结论。
    *   **原因**：需要基于原理进行逐步论证，CoT 能增强解释的连贯性和准确性。

**总结：**
对于**简单的事实性问答**（如“法国的首都是哪里？”），CoT 通常不会带来明显提升，甚至可能增加不必要的长度。但对于**需要“思考过程”才能得出正确答案**的复杂任务，CoT 是提升准确性的关键技术。


## 1.3 提示词模板

In [38]:
# 使用 f-string 构建提示词模板
topic = "Python 装饰器"
level = "初学者"

prompt = f"""请为{level}讲解{topic}。

要求：
1. 用简单的语言解释概念
2. 提供实际代码示例
3. 说明使用场景"""

response = client.chat([{"role": "user", "content": prompt}])
print(response)



# Python 装饰器入门指南（适合初学者）

## 一、什么是装饰器？（简单比喻）

想象你有一个函数，就像一件衣服。装饰器就像给这件衣服加一个“外套”或“标签”，在不修改衣服本身的情况下，增加新功能。

**装饰器的本质**：一个函数，用来包装另一个函数，增强它的功能。

---

## 二、装饰器的基本原理

装饰器通过“函数嵌套 + 返回函数”的方式实现。它接收一个函数作为参数，并返回一个新的函数，这个新函数在调用原函数前后可以执行额外逻辑。

---

## 三、代码示例（从简单到复杂）

### 1. 手动包装函数（理解原理）

```python
def say_hello():
    print("你好！")

def wrapper(func):
    def inner():
        print("开始执行...")
        func()
        print("执行结束！")
    return inner

say_hello = wrapper(say_hello)
say_hello()
```

输出：
```
开始执行...
你好！
执行结束！
```

### 2. 使用 `@` 语法糖（标准写法）

```python
def my_decorator(func):
    def wrapper():
        print("装饰器开始")
        func()
        print("装饰器结束")
    return wrapper

@my_decorator
def say_hello():
    print("你好！")

say_hello()
```

输出：
```
装饰器开始
你好！
装饰器结束
```

### 3. 带参数的函数装饰器

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"参数: {args}, {kwargs}")
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def greet(name, greeting="你好"):


---

# 第二章：角色设定

使用 System Prompt 设定 AI 的专业角色。

## 2.1 代码审查专家

In [39]:
messages = [
    {"role": "system", "content": "你是一位严格的代码审查专家，专注于 Python 代码质量和最佳实践。"},
    {"role": "user", "content": "请审查这段代码：\n\ndef get_user(id):\n    return db.query('SELECT * FROM users WHERE id = ' + id)"}
]
response = client.chat(messages)
print(response)



# 🔍 代码审查报告

## ⚠️ 严重问题

### 1. SQL 注入漏洞（高危）
```python
# ❌ 危险代码
db.query('SELECT * FROM users WHERE id = ' + id)
```
**风险**：攻击者可构造恶意输入，如 `id = "1 OR 1=1"`，导致数据泄露或数据库被篡改。

**修复方案**：使用参数化查询
```python
# ✅ 安全代码
db.query('SELECT * FROM users WHERE id = ?', (id,))
```

### 2. 缺少输入验证
- 未验证 `id` 的类型（应为整数）
- 未检查 `id` 是否为空或非法值

**修复方案**：
```python
if not isinstance(id, int) or id <= 0:
    raise ValueError("id 必须是正整数")
```

### 3. 缺少异常处理
- 数据库查询失败时未捕获异常
- 未处理用户不存在的情况

**修复方案**：
```python
try:
    result = db.query('SELECT * FROM users WHERE id = ?', (id,))
    if not result:
        raise UserNotFoundError(f"用户 {id} 不存在")
    return result
except DatabaseError as e:
    logger.error(f"数据库查询失败: {e}")
    raise
```

### 4. 缺少文档和类型提示
- 无函数文档字符串
- 无参数和返回值类型注解

**修复方案**：
```python
def get_user(id: int) -> User:
    """
    根据用户ID获取用户信息

    Args:
        id: 用户唯一标识符（正整数）

    Returns:
        User: 用户对象

    Raises:
        ValueError: 当id无效时
        UserNotFoundError: 当用户不存在时
        DatabaseError:

## 2.2 数学老师

In [40]:
messages = [
    {"role": "system", "content": "你是一位耐心的数学老师，擅长用循序渐进的方式讲解数学概念。"},
    {"role": "user", "content": "请解释什么是导数，以及它的几何意义。"}
]
response = client.chat(messages)
print(response)



你好！很高兴能为你解答这个问题。我是你的数学老师。

别担心，导数（Derivative）听起来可能有点抽象，但它其实是描述**“变化”**最有力的工具。我们不需要一开始就陷入复杂的公式，让我们像剥洋葱一样，一层一层地把它弄清楚。

准备好了吗？我们开始吧。

---

### 第一部分：导数是什么？（直观理解）

为了理解导数，我们先从生活中最熟悉的一个概念说起：**速度**。

想象你正在开车。
1.  **平均速度**：如果你从家开车到公司，一共走了 60 公里，用了 1 小时。那么你的平均速度是 $60 \div 1 = 60$ 公里/小时。
    *   在数学上，这叫做**平均变化率**。公式是：$\frac{\text{路程的变化量}}{\text{时间的变化量}}$。
2.  **瞬时速度**：但是，你在路上肯定有快有慢。有时候堵车只有 20 公里/小时，有时候在高速上能开到 100 公里/小时。当你经过某个特定的红绿灯路口时，你仪表盘上显示的那个数字，就是那一瞬间的速度。

**导数，本质上就是“瞬时变化率”。**

它回答的问题是：**当自变量（比如时间 $x$）发生极其微小的变化时，因变量（比如路程 $y$）变化的快慢是多少？**

#### 数学上的定义
如果我们有一个函数 $y = f(x)$，我们想知道在 $x$ 这个点，函数变化的快慢。
1.  我们先看一个小区间 $[x, x + \Delta x]$。
2.  在这个区间里，函数值的变化是 $\Delta y = f(x + \Delta x) - f(x)$。
3.  平均变化率就是 $\frac{\Delta y}{\Delta x}$。
4.  **导数**就是让 $\Delta x$ 无限趋近于 0 时，这个比值的极限。

用数学符号表示就是：
$$ f'(x) = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x} $$

别被这个公式吓到，它的核心思想就是：**把时间切片切得无限薄，看那一瞬间的变化率。**

---

### 第二部分：导数的几何意义（图形理解）

这是导数最直观、最重要的部分。让我们把函数画在坐标系里。

#### 1. 割线与切线
想象你在纸上画了一条弯曲的曲线（比如抛物线

## 2.3 面试官

In [41]:
messages = [
    {"role": "system", "content": "你是一位资深 Python 技术面试官，正在面试中级开发工程师。请提出有深度的问题。"},
    {"role": "user", "content": "我想了解 Python 的 GIL 机制，请提问。"}
]
response = client.chat(messages)
print(response)



你好。很高兴能和你探讨 Python 的核心机制。作为中级开发工程师，我不仅希望你了解 GIL 的“是什么”，更希望考察你对“为什么”以及“如何应对”的深度理解。

我们将通过四个层层递进的问题来进行。你可以尝试在脑海中构思答案，或者写下来。每道题后，我会附上**考察点**，帮助你自我评估。

---

### 问题一：底层原理与存在意义
**“我们都知道 CPython 解释器有一个全局解释器锁（GIL）。请从内存管理和线程安全的角度，解释一下为什么 CPython 需要引入 GIL？如果去掉 GIL，会对 Python 的内存模型产生什么影响？”**

> **🔍 面试官考察点：**
> *   **初级回答：** 只是说“为了线程安全”或“防止多线程同时执行”。
> *   **中级回答：** 能提到**引用计数（Reference Counting）**。解释因为 Python 对象的生命周期管理依赖引用计数，而引用计数的增减不是原子操作，GIL 保证了同一时刻只有一个线程修改引用计数，从而避免内存泄漏或崩溃。
> *   **高级回答：** 能进一步指出 GIL 是 CPython 实现的一种权衡（Trade-off）。去掉 GIL 需要引入更复杂的内存管理（如 GC 或原子操作），会显著增加解释器的复杂度和内存开销，且可能影响单线程性能。

---

### 问题二：调度机制与 I/O 行为
**“在多线程环境下，GIL 是如何在多个线程之间切换的？当线程执行 I/O 操作（如读写文件、网络请求）时，GIL 会发生什么变化？这与 `sys.setswitchinterval` 有什么关系？”**

> **🔍 面试官考察点：**
> *   **初级回答：** 知道 I/O 时会释放锁，CPU 计算时持有锁。
> *   **中级回答：** 能解释**检查间隔（Check Interval）**。GIL 不是永远不释放，而是每隔一定时间（默认 5ms，可通过 `sys.setswitchinterval` 调整）强制切换。
> *   **高级回答：** 能区分**阻塞型 I/O**和**非阻塞型 I/O**。在阻塞 I/O 发生时，线程会主动释放 GIL 等待，其他线程可获取 GIL 执行。同时能提到 C 扩展（C Extensions）中如果长时

---

# 第三章：Few-Shot 学习

通过提供示例来引导模型理解任务。

## 3.1 文本分类

In [42]:
few_shot_prompt = """请将以下评论分类为：正面、中性、负面。

示例 1:
评论：这个产品太棒了，质量非常好！
分类：正面

示例 2:
评论：产品还可以，但价格有点贵。
分类：中性

示例 3:
评论：完全不符合预期，非常失望。
分类：负面

待分类:
评论：功能基本满足需求，客服响应速度有待提高。
分类："""

response = client.chat([{"role": "user", "content": few_shot_prompt}])
print(response)



中性


## 3.2 信息提取

In [43]:
extraction_prompt = """从文本中提取实体信息，输出 JSON 格式。

示例 1:
输入：张三于 2023 年加入腾讯公司，担任软件工程师。
输出：{"person": "张三", "year": "2023", "company": "腾讯", "position": "软件工程师"}

示例 2:
输入：李四在 2020 年毕业于北京大学计算机系。
输出：{"person": "李四", "year": "2020", "company": "北京大学", "position": "计算机系"}

待处理:
输入：王五于 2022 年入职阿里巴巴，职位是数据科学家。
输出："""

response = client.chat([{"role": "user", "content": extraction_prompt}])
print(response)



{"person": "王五", "year": "2022", "company": "阿里巴巴", "position": "数据科学家"}


## 3.3 SQL 生成

In [44]:
sql_prompt = """将自然语言转换为 SQL 查询。

表结构：students(id, name, age, major, gpa)

示例 1:
输入：找出所有计算机专业的学生
输出：SELECT * FROM students WHERE major = '计算机';

示例 2:
输入：按绩点降序排列前 10 名学生
输出：SELECT * FROM students ORDER BY gpa DESC LIMIT 10;

待转换:
输入：找出年龄大于 20 且绩点超过 3.5 的学生
输出："""

response = client.chat([{"role": "user", "content": sql_prompt}])
print(response)



SELECT * FROM students WHERE age > 20 AND gpa > 3.5;


---

# 第四章：思维链 (CoT)

引导模型展示推理过程，提高复杂问题的准确性。

## 4.1 零样本 CoT

In [45]:
problem = """
问题：一个水池有进水管和出水管。进水管每分钟注水 15 升，出水管每分钟排水 8 升。
      如果水池容量是 500 升，初始为空，同时打开进水管和出水管，需要多少分钟才能装满？
"""

# 添加"请一步一步思考"
cot_prompt = problem + "\n请一步一步思考，展示完整的推理过程。"

response = client.chat([{"role": "user", "content": cot_prompt}])
print(response)



这是一个经典的工程问题（注水问题）。我们可以按照以下步骤进行推理和计算：

### 第一步：分析已知条件
*   **进水管速度**：每分钟注入 15 升水。
*   **出水管速度**：每分钟排出 8 升水。
*   **水池容量**：500 升。
*   **初始状态**：水池是空的（0 升）。
*   **操作方式**：同时打开进水管和出水管。

### 第二步：计算实际的净注水速度
因为进水管和出水管是同时工作的，水池中水量的实际增加速度取决于进水速度减去排水速度。

*   **净注水速度** = 进水速度 - 排水速度
*   **计算**：$15 \text{ 升/分钟} - 8 \text{ 升/分钟} = 7 \text{ 升/分钟}$

这意味着，虽然每分钟有 15 升水进来，但同时也流走了 8 升，所以水池里每分钟实际只增加了 7 升水。

### 第三步：计算装满水池所需的时间
我们要计算填满 500 升容量需要多少分钟，可以用总容量除以净注水速度。

*   **所需时间** = 水池总容量 ÷ 净注水速度
*   **计算**：$500 \div 7$

### 第四步：执行除法运算
*   $500 \div 7 = 71 \dots 3$
*   即：$71$ 分钟还余 $3$ 升。
*   用分数表示为：$71 \frac{3}{7}$ 分钟。
*   用小数表示约为：$71.43$ 分钟。

如果需要换算成分钟和秒：
*   余下的 $\frac{3}{7}$ 分钟 $\times 60$ 秒/分钟 $\approx 25.7$ 秒。

### 第五步：得出结论
水池装满需要的时间是 $71 \frac{3}{7}$ 分钟（或约 71.43 分钟）。

**最终答案：**
需要 **$71 \frac{3}{7}$** 分钟（或约 **71.43** 分钟）才能装满。


## 4.2 数学应用题 - 完整 CoT

In [46]:
math_problem = """
某电商平台在双十一进行促销：
- 原价商品打 8 折
- 优惠券：满 200 减 50
- 购物节额外补贴：最终价格的 10%

小明想买一个原价 450 元的背包，实际需要支付多少元？
"""

cot_steps = """请按照以下步骤逐步推理：

**步骤 1**: 计算折后价格
**步骤 2**: 判断是否满足优惠券条件
**步骤 3**: 计算优惠后的价格
**步骤 4**: 计算购物节补贴金额
**步骤 5**: 计算最终支付金额

每个步骤都要写出计算公式和结果。"""

response = client.chat([{"role": "user", "content": math_problem + cot_steps}])
print(response)



**步骤 1**: 计算折后价格
*   **计算公式**: 原价 × 0.8
*   **计算过程**: $450 \times 0.8 = 360$
*   **结果**: 360 元

**步骤 2**: 判断是否满足优惠券条件
*   **计算公式**: 折后价格 ≥ 200
*   **计算过程**: $360 \ge 200$
*   **结果**: 满足条件（可以使用优惠券）

**步骤 3**: 计算优惠后的价格
*   **计算公式**: 折后价格 - 优惠券减免额
*   **计算过程**: $360 - 50 = 310$
*   **结果**: 310 元

**步骤 4**: 计算购物节补贴金额
*   **计算公式**: 优惠后的价格 × 10%
*   **计算过程**: $310 \times 10\% = 31$
*   **结果**: 31 元

**步骤 5**: 计算最终支付金额
*   **计算公式**: 优惠后的价格 - 购物节补贴金额
*   **计算过程**: $310 - 31 = 279$
*   **结果**: 279 元

**最终答案**: 小明实际需要支付 **279** 元。


## 4.3 逻辑推理 - CoT + 自我验证

In [47]:
logic_problem = """
判断以下论证是否正确：
"如果所有猫都喜欢鱼，小明是一只猫，所以小明喜欢吃鱼。"
"""

verification_prompt = """请按以下步骤分析：

**步骤 1**: 识别前提和结论
**步骤 2**: 检查前提是否为真
**步骤 3**: 检查推理形式是否有效
**步骤 4**: 得出结论

**额外验证**: 尝试构造一个反例来检验论证的有效性。"""

response = client.chat([{"role": "user", "content": logic_problem + verification_prompt}])
print(response)



**步骤 1: 识别前提和结论**

*   **前提 1**：所有猫都喜欢鱼。
*   **前提 2**：小明是一只猫。
*   **结论**：小明喜欢吃鱼。

**步骤 2: 检查前提是否为真**

*   **前提 1 的真假**：在现实世界中，“所有猫都喜欢鱼”是一个全称判断。事实上，并非每一只猫都喜欢鱼（有的猫可能过敏、挑食或更喜欢肉类），因此该前提在事实层面上是**不成立**的（或至少是存疑的过度概括）。
*   **前提 2 的真假**：“小明”通常是人类的姓名。除非特指一只名叫“小明”的宠物猫，否则在常规语境下，小明是人而不是猫。因此，该前提在事实层面上通常也是**假**的。
*   **小结**：该论证的两个前提在现实事实中均难以成立。

**步骤 3: 检查推理形式是否有效**

*   **逻辑形式**：这是一个典型的三段论推理。
    *   大前提：所有 M（猫）都是 P（喜欢鱼）。
    *   小前提：S（小明）是 M（猫）。
    *   结论：S（小明）是 P（喜欢鱼）。
*   **有效性分析**：从形式逻辑的角度来看，这种推理结构（Barbara 式）是**有效**的。如果前提为真，结论必然为真。
*   **潜在漏洞**：需要注意前提中的“喜欢鱼”与结论中的“喜欢吃鱼”在语义上是否完全等同。如果“喜欢鱼”指喜欢玩弄鱼，而“喜欢吃鱼”指食用，则犯了“偷换概念”（四词项谬误）的错误。但在常规逻辑练习中，通常默认二者概念一致。在此假设概念一致的情况下，推理形式是有效的。

**步骤 4: 得出结论**

*   **逻辑有效性**：该论证的**推理形式是正确的（有效的）**。
*   **论证可靠性**：由于前提在事实上不成立，该论证是**不可靠的（Unsound）**。这意味着虽然推导过程没有逻辑错误，但无法保证结论在现实中为真。
*   **综合判断**：如果仅评价逻辑推导过程，论证是**正确**的；如果评价结论的真实性，论证是**不成立**的。

---

**额外验证：尝试构造一个反例来检验论证的有效性**

为了检验论证的**有效性**（Validity），我们需要尝试构造一个反例，即：**前提全部为真，但结论为假**的情况。如果存在这种情况，则论证无效；如果不存在，则论证有效。

*   **尝试构造**：
 

---

# 第五章：ReAct 模式

Reasoning + Acting：结合推理与行动，适用于需要外部工具的场景。

In [48]:
# ReAct 风格 Prompt
react_prompt = """一个商店销售以下商品：
- iPhone 15: 原价 7999 元，促销价 6999 元
- AirPods Pro: 原价 1999 元，打 8 折
- iPad Air: 原价 4799 元，参加满 1000 减 200 活动

问题：小明买了 1 部 iPhone 和 2 个 AirPods，总价多少？

请按以下格式推理:

Thought: 分析需要计算的内容
Action: calculate <计算表达式>
Observation: <计算结果>
...重复直到得到答案
Final Answer: <最终总价>"""

response = client.chat([{"role": "user", "content": react_prompt}])
print(response)



Thought: 确定 iPhone 15 的购买价格，根据促销信息为 6999 元。
Action: calculate 6999
Observation: 6999

Thought: 计算单个 AirPods Pro 的折后价格，原价 1999 元打 8 折。
Action: calculate 1999 * 0.8
Observation: 1599.2

Thought: 计算 2 个 AirPods Pro 的总价。
Action: calculate 1599.2 * 2
Observation: 3198.4

Thought: 计算所有商品的最终总价。
Action: calculate 6999 + 3198.4
Observation: 10197.4

Final Answer: 10197.4 元


## 5.1 ReAct Agent 类

In [49]:
class ReActAgent:
    """改进的 ReAct Agent - 支持多工具调用"""
    
    def __init__(self, client):
        self.client = client
        self.tools = {}
    
    def register_tool(self, name, func):
        self.tools[name] = func
    
    def run(self, task, max_iterations=5):
        tool_desc = "\n".join([f"- {name}: {func.__doc__}" for name, func in self.tools.items()])
        
        system_prompt = f"""你是一个推理和行动 Agent。

可用工具:
{tool_desc}

输出格式规则:
1. Thought: 先写下你的思考过程
2. Action: 调用工具，格式为 "Action: <工具名> <参数>"
3. 如果需要多个工具，列出所有 Action（每行一个）
4. 如果已有足够信息，输出 "Final Answer: <答案>"

重要：不要自己编写 Observation，系统会自动返回工具执行结果！"""
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"任务：{task}"}
        ]
        
        for i in range(max_iterations):
            response = self.client.chat(messages)
            print(f"\n--- 迭代 {i+1} ---")
            print(response)
            
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip()
            
            # 解析并执行所有 Action
            observations = []
            action_lines = [line for line in response.split("\n") if line.strip().startswith("Action:")]
            
            if not action_lines:
                print("未找到 Action，尝试继续...")
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": "请按照格式输出 Action。"})
                continue
            
            for line in action_lines:
                parts = line.replace("Action:", "").strip().split(maxsplit=1)
                if len(parts) == 2:
                    tool_name, tool_arg = parts
                    if tool_name in self.tools:
                        result = self.tools[tool_name](tool_arg.strip('"\''))
                        observations.append(f"[{tool_name} {tool_arg}]: {result}")
                        print(f"  → {tool_name}({tool_arg}) = {result}")
                    else:
                        observations.append(f"[{tool_name}]: 工具不存在")
            
            if observations:
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"工具执行结果:\n" + "\n".join(observations)})
            else:
                return "未能执行任何工具"
        
        return "超过最大迭代次数，未能得到答案"

print("ReActAgent 类定义完成")

ReActAgent 类定义完成


In [50]:
# 使用 ReAct Agent
agent = ReActAgent(client)

# 注册天气查询工具
def get_weather(city):
    """获取城市天气"""
    weather_db = {"北京": "晴，25°C", "上海": "多云，28°C", "深圳": "雷阵雨，30°C"}
    return weather_db.get(city, f"未找到{city}的天气信息")

agent.register_tool("weather", get_weather)

result = agent.run("北京和上海哪个城市更热？", max_iterations=3)
print(f"\n最终结果：{result}")


--- 迭代 1 ---


Thought: 用户想知道北京和上海哪个城市更热，我需要获取这两个城市的天气信息来进行温度比较。我需要分别查询北京和上海的天气数据。

Action: weather 北京
Action: weather 上海
  → weather(北京) = 晴，25°C
  → weather(上海) = 多云，28°C

--- 迭代 2 ---


Final Answer: 上海更热，当前温度为28°C，而北京为25°C。

最终结果：上海更热，当前温度为28°C，而北京为25°C。


---

# 第六章：高级技巧

Self-Consistency、Tree of Thoughts、Prompt Chaining

## 6.1 Self-Consistency 自洽性

In [52]:
def extract_chicken_rabbit_answer(text):
    """从模型回答中抽取“鸡/兔”的数量，用于自洽性投票。"""
    if not text:
        return None

    final_part = text.split("最终答案")[-1] if "最终答案" in text else text

    ordered_patterns = [
        (r"鸡\s*=\s*(\d+).*?兔\s*=\s*(\d+)", "chicken_first"),
        (r"兔\s*=\s*(\d+).*?鸡\s*=\s*(\d+)", "rabbit_first"),
        (r"鸡[^0-9]{0,8}(\d+)\s*只.*?兔[^0-9]{0,8}(\d+)\s*只", "chicken_first"),
        (r"兔[^0-9]{0,8}(\d+)\s*只.*?鸡[^0-9]{0,8}(\d+)\s*只", "rabbit_first"),
    ]

    def parse_from(target):
        for pattern, order in ordered_patterns:
            matches = list(re.finditer(pattern, target, flags=re.S))
            if not matches:
                continue
            first, second = map(int, matches[-1].groups())
            if order == "chicken_first":
                return {"鸡": first, "兔": second}
            return {"鸡": second, "兔": first}
        return None

    return parse_from(final_part) or parse_from(text)


def self_consistency(client, prompt, question, n_samples=3):
    """多次采样，抽取最终答案，并选择出现次数最多的答案。"""
    samples = []
    answer_votes = []

    for i in range(n_samples):
        full_prompt = f"""{prompt}

问题：{question}

请先简要列式推理，最后单独用一行输出：
最终答案：鸡=数字，兔=数字"""

        response = client.chat(
            [{"role": "user", "content": full_prompt}],
            temperature=0.7,
            max_tokens=4096,
        )

        print(f"\n采样 {i+1}/{n_samples}:")
        if not response:
            print("本次采样未返回有效内容，已跳过。")
            continue

        print(response)
        samples.append(response)

        parsed = extract_chicken_rabbit_answer(response)
        if parsed:
            answer_votes.append((parsed["鸡"], parsed["兔"]))
        else:
            print("未能从本次回答中抽取标准答案。")

    if not samples:
        raise RuntimeError("所有采样都没有返回有效内容，请检查 LLMClient 配置、max_tokens 或 API 返回格式。")

    if not answer_votes:
        print("未抽取到可投票答案，返回第一条有效回答。")
        return samples[0]

    vote_counter = Counter(answer_votes)
    best_answer, count = vote_counter.most_common(1)[0]

    print(f"\n共获得 {len(samples)} 条有效回答，抽取到 {len(answer_votes)} 个可投票答案。")
    print(f"投票结果：{dict(vote_counter)}")

    chicken, rabbit = best_answer
    return f"鸡={chicken}，兔={rabbit}（{count}/{len(answer_votes)} 票）"


# 测试
problem = "鸡兔同笼问题：共有 35 个头，94 只脚，问鸡和兔各多少只？"
prompt = "请用代数方法推理这个问题。"

result = self_consistency(client, prompt, problem, n_samples=3)
print(f"\n最终答案：{result}")


采样 1/3:


设鸡有 $x$ 只，兔有 $y$ 只。
根据题意列出方程组：
1. 头数关系：$x + y = 35$
2. 脚数关系：$2x + 4y = 94$

解方程组：
由方程 1 得 $x = 35 - y$，代入方程 2：
$2(35 - y) + 4y = 94$
$70 - 2y + 4y = 94$
$2y = 24$
$y = 12$

将 $y = 12$ 代入 $x = 35 - y$：
$x = 35 - 12 = 23$

所以鸡有 23 只，兔有 12 只。

最终答案：鸡=23，兔=12

采样 2/3:


设鸡有 $x$ 只，兔有 $y$ 只。根据题意可列出二元一次方程组：

1.  头的数量：$x + y = 35$
2.  脚的数量：$2x + 4y = 94$

解方程组：
由方程 (1) 得 $x = 35 - y$，将其代入方程 (2)：
$$2(35 - y) + 4y = 94$$
$$70 - 2y + 4y = 94$$
$$2y = 24$$
$$y = 12$$

将 $y = 12$ 代入 $x = 35 - y$ 得：
$$x = 35 - 12 = 23$$

最终答案：鸡=23，兔=12

采样 3/3:


设鸡的数量为 $x$，兔的数量为 $y$。
根据头数可得方程：$x + y = 35$
根据脚数可得方程：$2x + 4y = 94$

解方程组：
由第一个方程得 $x = 35 - y$，代入第二个方程：
$2(35 - y) + 4y = 94$
$70 - 2y + 4y = 94$
$2y = 24$
$y = 12$

将 $y=12$ 代入 $x = 35 - y$：
$x = 23$

最终答案：鸡=23，兔=12

共获得 3 条有效回答，抽取到 3 个可投票答案。
投票结果：{(23, 12): 3}

最终答案：鸡=23，兔=12（3/3 票）


## 6.2 Tree of Thoughts 思维树

In [53]:
def tree_of_thought(client, problem, options):
    """探索多种解题路径"""
    results = []
    
    for i, option in enumerate(options):
        prompt = f"""问题：{problem}

考虑方案 {i+1}: {option}

请分析:
1. 优点（3-5 点）
2. 缺点或风险（2-3 点）
3. 适用场景
4. 评分 1-10 分"""
        
        response = client.chat([{"role": "user", "content": prompt}])
        results.append({"option": option, "analysis": response})
        print(f"\n方案 {i+1} 分析完成")
    
    # 综合评估
    options_summary = "\n".join([f'方案{i+1}: {r["option"]}' for i, r in enumerate(results)])
    analysis_summary = "\n".join([f'【方案{i+1}】\n{r["analysis"]}' for i, r in enumerate(results)])
    
    synthesis_prompt = f"""请比较以下方案并给出推荐：

{options_summary}

分析如下:
{analysis_summary}

请给出最佳推荐及理由。"""
    
    final = client.chat([{"role": "user", "content": synthesis_prompt}])
    return final

# 测试
tech_problem = "创业公司应该选择什么技术栈来构建 MVP？"
options = ["微服务 +Kubernetes", "单体架构 + 模块化", "Serverless+BaaS"]

result = tree_of_thought(client, tech_problem, options)
print("\n" + "="*50)
print("综合推荐")
print("="*50)
print(result)


方案 1 分析完成

方案 2 分析完成

方案 3 分析完成

综合推荐


基于您提供的详细分析，以下是针对创业公司 MVP 阶段的最终对比总结与推荐建议。

### 🏆 最终推荐：方案 2（单体架构 + 模块化）

**推荐指数：⭐⭐⭐⭐⭐ (首选)**
**备选方案：方案 3（Serverless + BaaS）**
**坚决避免：方案 1（微服务 + Kubernetes）**

---

### 1. 核心对比总结

| 维度 | 方案 1：微服务 + K8s | 方案 2：单体 + 模块化 | 方案 3：Serverless + BaaS |
| :--- | :--- | :--- | :--- |
| **开发速度** | 🐢 慢 (复杂度高) | 🐇 快 (流程简单) | 🚀 极快 (无需运维) |
| **初期成本** | 💰 高 (资源 + 人力) | 💵 低 (常规服务器) | 🆓 极低 (免费额度) |
| **运维难度** | 🔥 极高 (需专家) | 🟢 低 (常规运维) | 🟢 极低 (云厂商托管) |
| **技术掌控力** | 🔒 高 (完全自主) | 🔒 高 (完全自主) | 🔓 低 (厂商锁定风险) |
| **扩展性** | 🚀 极强 (独立扩缩容) | 🐢 中等 (整体扩容) | 🚀 自动弹性 |
| **MVP 适配度** | ❌ 过度设计 | ✅ **最佳平衡** | ✅ **极速验证** |

---

### 2. 为什么首选方案 2（单体架构 + 模块化）？

虽然方案 3 在速度上略胜一筹，但**方案 2 是大多数创业公司长期生存的最优解**，理由如下：

1.  **避免“成功后的技术债”**：
    *   方案 3 (BaaS) 最大的隐患是**厂商锁定**。如果 MVP 验证成功，用户量激增，迁移出 Firebase/Supabase 等平台的成本极高（数据格式、API 逻辑、鉴权体系均需重构）。
    *   方案 2 的代码和数据库完全掌握在自己手中，未来无论业务如何增长，都可以平滑演进为微服务，没有迁移包袱。

2.  **招聘与协作更友好**：
    *   市场上熟悉传统后端（Java/Go/Node + SQL）的工程师远多于精通特定 BaaS 生态的专家。
    *   单体架

## 6.3 Prompt Chaining 提示链

In [60]:
def safe_chat(client, messages, retries=2, **kwargs):
    """稳定调用 LLM：显式关闭 thinking，并兼容旧 kernel 中缓存的 LLMClient。"""
    import re
    from openai import OpenAI

    call_kwargs = {
        "temperature": 0.3,
        "max_tokens": 2048,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    }
    call_kwargs.update(kwargs)

    def clean(text):
        if not text:
            return ""
        return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.S).strip()

    def call_once(current_messages, current_kwargs):
        # 直接调用 Qwen/OpenAI 兼容接口，避免旧版 client.chat 没传 extra_body。
        if getattr(client.config, "provider", None) == "qwen":
            raw_client = OpenAI(api_key=client.config.api_key, base_url=client.config.base_url)
            response = raw_client.chat.completions.create(
                model=client.config.model,
                messages=current_messages,
                temperature=current_kwargs["temperature"],
                max_tokens=current_kwargs["max_tokens"],
                extra_body=current_kwargs["extra_body"],
            )
            choice = response.choices[0]
            content = clean(choice.message.content)
            if content:
                return content
            print(f"  ! finish_reason={choice.finish_reason}, content 为空。")
            return ""

        return clean(client.chat(current_messages, **current_kwargs))

    prompt_variants = [messages]
    if messages and messages[-1].get("role") == "user":
        concise_messages = list(messages)
        concise_messages[-1] = {
            "role": "user",
            "content": messages[-1]["content"] + "\n\n请用最简洁的中文直接回答，控制在 150 字以内。",
        }
        prompt_variants.append(concise_messages)

    for variant_index, current_messages in enumerate(prompt_variants, start=1):
        for attempt in range(retries + 1):
            result = call_once(current_messages, call_kwargs)
            if result:
                return result
            print(f"  ! 变体 {variant_index} 第 {attempt + 1} 次调用返回空内容，准备重试...")

    return ""


def compact_context(results, max_chars=900):
    """只保留前面步骤的必要信息，避免链式上下文越来越长。"""
    blocks = []
    for item in results:
        text = item["result"] or ""
        if len(text) > max_chars:
            text = text[:max_chars] + "\n...（已截断）"
        blocks.append(f"[步骤 {item['step']} 结果]\n{text}")
    return "\n\n".join(blocks) if blocks else "无"


def local_summary(results):
    """模型临时空返回时的本地兜底摘要，保证链路不返回 None。"""
    joined = " ".join((item["result"] or "") for item in results)
    keywords = ["Prompt Engineering", "角色设定", "Few-Shot", "思维链", "指令优化", "上下文", "大模型"]
    seen = [kw for kw in keywords if kw.lower() in joined.lower()]
    if seen:
        return "提示工程通过" + "、".join(seen[:5]) + "等方法组织指令和上下文，引导大模型更准确地理解任务、按要求输出，并提升复杂问题的推理与生成质量。"
    return "提示工程通过明确任务、提供上下文、加入示例和拆解步骤，引导大模型生成更准确、稳定、符合格式要求的结果。"


def prompt_chaining(client, task, chain):
    """将复杂任务拆成多个子任务，并把每一步的有效结果传给下一步。"""
    results = []

    for i, step in enumerate(chain):
        context = compact_context(results)
        step_prompt = f"""【步骤 {i+1}: {step['task']}】

总任务：{task}
当前任务：{step['description']}

已有步骤结果：
{context}

执行要求：
{step['prompt']}

请直接输出本步骤结果，不要输出思考过程。"""

        result = safe_chat(
            client,
            [{"role": "user", "content": step_prompt}],
            retries=1,
            max_tokens=2048,
        )

        if not result and step["task"] == "摘要生成":
            print("  ! 摘要生成调用为空，使用本地兜底摘要。")
            result = local_summary(results)

        if not result:
            fallback_prompt = f"""请完成以下单个任务，不要输出思考过程。

总任务：{task}
当前步骤：{step['task']}
任务说明：{step['description']}
执行要求：{step['prompt']}

可参考的已有结果：
{context[:700]}"""
            print(f"  ! 步骤 {i + 1} 使用压缩降级 prompt 重试。")
            result = safe_chat(
                client,
                [{"role": "user", "content": fallback_prompt}],
                retries=1,
                max_tokens=1024,
            )

        if not result:
            result = "[本步骤未获得模型返回，请稍后单独重试该步骤。]"

        results.append({"step": step["task"], "result": result})
        print(f"✓ 步骤 {i + 1} 完成：{step['task']}")

    return results


# 测试：文章分析链
chain = [
    {
        "task": "关键词提取",
        "description": "提取核心关键词",
        "prompt": "请从以下主题中提取 5-10 个核心关键词：Prompt Engineering 的核心技术包括角色设定、Few-Shot 学习、思维链等。",
    },
    {
        "task": "结构分析",
        "description": "分析结构",
        "prompt": "基于上一步关键词，分析内容的结构和主要论点。",
    },
    {
        "task": "摘要生成",
        "description": "生成摘要",
        "prompt": "基于前两步结果，用 100 字总结核心内容。",
    },
]

results = prompt_chaining(client, "内容分析", chain)

print("\n" + "=" * 50)
print("最终结果")
print("=" * 50)
for r in results:
    print(f"\n### {r['step']}")
    print(r["result"])

✓ 步骤 1 完成：关键词提取
✓ 步骤 2 完成：结构分析
✓ 步骤 3 完成：摘要生成

最终结果

### 关键词提取
Prompt Engineering、角色设定、Few-Shot 学习、思维链、核心技术、提示工程、大模型、上下文学习、推理能力、任务优化

### 结构分析
### 结构分析结果

基于提取的关键词，该内容呈现出"**核心定义—技术支柱—进阶策略—价值目标**"的层进式逻辑结构，旨在系统阐述提示工程（Prompt Engineering）如何优化大模型表现。

#### 1. 核心定义与基础（基石层）
*   **核心概念**：以**提示工程**为总领，明确其作为与大模型交互的核心技术定位。
*   **基础要素**：强调**角色设定**是构建有效提示的基础，通过赋予模型特定身份来规范输出风格与逻辑边界。

#### 2. 关键技术支柱（方法层）
内容围绕提升模型能力的三大核心技术展开：
*   **上下文学习**：通过**Few-Shot 学习**（少样本学习），利用少量示例在上下文中引导模型理解任务模式，无需参数微调即可适应新任务。
*   **推理增强**：引入**思维链**（Chain of Thought）技术，通过拆解复杂问题为中间推理步骤，显著提升模型的逻辑推理与复杂任务处理能力。
*   **技术整合**：将上述方法统合为**核心技术**体系，形成标准化的操作范式。

#### 3. 价值目标与成效（应用层）
*   **最终导向**：所有技术手段均服务于**任务优化**这一核心目标。
*   **能力跃迁**：通过结构化提示，实现从简单指令执行到复杂**推理能力**的跨越，最大化**大模型**的潜在效能。

#### 主要论点总结
1.  **提示工程是解锁大模型能力的关键钥匙**，而非简单的指令输入。
2.  **结构化策略优于直觉式提问**：通过角色设定、少样本示例和思维链的有机结合，能系统性解决模型幻觉与逻辑缺失问题。
3.  **上下文即能力**：利用上下文学习（Few-Shot）和思维链，可以在不改变模型参数的情况下，动态提升模型在特定任务上的推理精度与适应性。

### 摘要生成
提示工程是优化大模型的关键，通过角色设定奠定基础，结合少样本学习与思维链技术，构建结构化提示体系。该体系在不微调参数下，显著提升模型

---

# 第七章：RAG 系统

检索增强生成 (Retrieval-Augmented Generation)

In [67]:
@dataclass
class Document:
    """文档类"""
    content: str
    metadata: Dict = field(default_factory=dict)

@dataclass
class RetrievedChunk:
    """检索结果"""
    content: str
    score: float
    metadata: Dict

class SimpleRAG:
    """简化的 RAG 系统"""
    
    def __init__(self, client):
        self.client = client
        self.documents: List[Document] = []
    
    def add_documents(self, docs: List[Document]):
        self.documents.extend(docs)
        print(f"已添加 {len(docs)} 个文档")
    
    def retrieve(self, query: str, top_k: int = 3) -> List[RetrievedChunk]:
        """关键词检索"""
        query_words = self._tokenize(query.lower())
        results = []
        
        for doc in self.documents:
            doc_words = self._tokenize(doc.content.lower())
            intersection = query_words & doc_words
            if intersection:
                score = len(intersection) / max(len(query_words), 1)
                results.append(RetrievedChunk(content=doc.content, score=score, metadata=doc.metadata))
        
        results.sort(key=lambda x: x.score, reverse=True)
        return results[:top_k]
    
    def _tokenize(self, text: str) -> set:
        """简单分词"""
        english_words = set(re.findall(r'[a-zA-Z]+', text.lower()))
        chinese_chars = set(re.findall(r'[\u4e00-\u9fff]+', text))
        chinese_words = set()
        for chars in chinese_chars:
            for i in range(len(chars)):
                for j in range(i+2, min(i+5, len(chars)+1)):
                    chinese_words.add(chars[i:j])
        return english_words | chinese_words
    
    def generate_answer(self, query: str, context_docs: List[RetrievedChunk]) -> str:
        context = "\n\n".join([f"【文档{i+1}】{doc.content}" for i, doc in enumerate(context_docs)])
        
        prompt = f"""基于以下资料回答问题：

【问题】{query}

【资料】
{context}

【回答】"""
        
        return self.client.chat([{"role": "user", "content": prompt}])
    
    def query(self, question: str, top_k: int = 3) -> str:
        docs = self.retrieve(question, top_k)
        if not docs:
            return "知识库中没有找到相关信息。"
        return self.generate_answer(question, docs)

print("SimpleRAG 类定义完成")

SimpleRAG 类定义完成


In [68]:
# RAG 系统演示
rag = SimpleRAG(client)

# 添加知识库文档
docs = [
    Document(content="Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。特点是简洁易读的语法。", metadata={"source": "python_basics"}),
    Document(content="FastAPI 是一个现代 Python Web 框架，基于 Pydantic 和 Starlette，支持异步编程。", metadata={"source": "fastapi"}),
    Document(content="Django 是 Python 全栈框架，包含 ORM、管理后台、认证系统。适合大型 CMS。", metadata={"source": "django"}),
    Document(content="Docker 是容器化平台，可以打包应用及其依赖，实现一致的运行环境。", metadata={"source": "docker"}),
]
rag.add_documents(docs)

# 问答
questions = ["Python 语言有什么特点？", "FastAPI 和 Django 有什么区别？", "hermes agent 是什么？"]

for q in questions:
    print(f"\n问题：{q}")
    print("-" * 40)
    answer = rag.query(q)
    print(f"回答：{answer}")

已添加 4 个文档

问题：Python 语言有什么特点？
----------------------------------------
回答：

根据提供的资料，Python 语言的特点如下：

1.  **高级编程语言**：Python 被定义为一种高级编程语言（来源：文档 1）。
2.  **语法简洁易读**：其显著特点是拥有简洁且易读的语法（来源：文档 1）。

（注：文档 2 和文档 3 主要介绍了基于 Python 的框架 FastAPI 和 Django，未直接描述 Python 语言本身的核心特点。）

问题：FastAPI 和 Django 有什么区别？
----------------------------------------
回答：

基于提供的资料，FastAPI 和 Django 的区别如下：

1.  **框架定位**：FastAPI 是一个现代 Python Web 框架；Django 是 Python 全栈框架。
2.  **技术基础**：FastAPI 基于 Pydantic 和 Starlette。
3.  **功能特性**：FastAPI 支持异步编程；Django 包含 ORM、管理后台、认证系统。
4.  **适用场景**：Django 适合大型 CMS。

问题：hermes agent 是什么？
----------------------------------------
回答：知识库中没有找到相关信息。


---

# 第八章：教学案例

实际教学场景中的应用

## 8.1 智能代码审查

In [63]:
code_review_prompt = """你是一位严格的代码审查专家。

【任务背景】数据结构课程作业，实现快速排序。

【评分标准】
1. 正确性 (40 分)
2. 效率 (30 分)
3. 代码规范 (20 分)
4. 创新性 (10 分)

【待审查代码】
```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left = [i for i in arr if i < pivot]
    mid = [i for i in arr if i == pivot]
    right = [i for i in arr if i > pivot]
    return quicksort(left) + mid + quicksort(right)
```

请给出审查结果和总分。"""

response = client.chat([{"role": "user", "content": code_review_prompt}])
print(response)



# 代码审查报告 (Code Review Report)

**审查对象**：快速排序 (Quick Sort) 实现
**审查人**：代码审查专家
**审查日期**：2023-10-27
**总体评价**：代码逻辑正确，风格简洁，但严重违背了数据结构课程中关于“快速排序”的核心教学目标（原地排序与空间复杂度）。

---

## 1. 分项评分详情

### 1.1 正确性 (Correctness) - 40 / 40
*   **评分理由**：算法逻辑完全正确。
    *   基准情况（Base Case）处理得当（`len(arr) <= 1`）。
    *   分区逻辑（Partitioning）能够正确处理小于、等于、大于基准值的元素。
    *   能够正确处理重复元素（通过 `mid` 列表）。
    *   递归终止条件明确，不会死循环。
*   **扣分项**：无。

### 1.2 效率 (Efficiency) - 10 / 30
*   **评分理由**：**严重扣分项**。虽然时间复杂度平均为 $O(n \log n)$，但空间复杂度极差。
    *   **空间复杂度**：该实现每次递归都创建新的列表 (`left`, `mid`, `right`)。在最坏情况下（如数组已排序），递归深度为 $O(n)$，每层空间开销为 $O(n)$，总空间复杂度达到 **$O(n^2)$**。而标准的快速排序应为原地排序，空间复杂度为 **$O(\log n)$**（栈空间）。
    *   **内存分配**：频繁的列表创建和拼接 (`+`) 操作会产生大量的临时对象，导致 GC 压力增大，实际运行效率远低于原地交换实现。
    *   **课程目标偏离**：在数据结构课程中，快速排序的核心优势在于“原地排序 (In-place Sorting)"，此实现完全丢失了这一特性。
*   **扣分项**：空间复杂度未达标 (-15)，内存分配开销大 (-5)。

### 1.3 代码规范 (Code Conventions) - 15 / 20
*   **评分理由**：代码可读性尚可，但缺乏工程化规范。
    *   **优点**：变量命名清晰 (`pivot`, `left`, `right`)，符合 PEP 8 基本格式。
 

## 8.2 算法可视化解释

In [64]:
algo_visualizer_prompt = """你是一个算法可视化教学助手。

请解释"冒泡排序"如何对数组 [5, 2, 8, 1, 9] 进行排序。

要求：
1. 分步展示每一轮的比较和交换
2. 用图示表示数组状态变化
3. 标注已排序部分
4. 说明时间复杂度"""

response = client.chat([{"role": "user", "content": algo_visualizer_prompt}])
print(response)



你好！我是你的算法可视化教学助手。很高兴为你讲解 **冒泡排序 (Bubble Sort)**。

冒泡排序的核心思想是：**重复地走访过要排序的元素列，依次比较两个相邻的元素，如果顺序错误就把他们交换过来。** 就像水底的气泡一样，每一轮都会把当前未排序部分中**最大**的元素“冒泡”到末尾。

我们将对数组 `[5, 2, 8, 1, 9]` 进行升序排序。

---

### 🟢 初始状态
```text
[ 5, 2, 8, 1, 9 ]
 ↑ 未排序区域 (全部)
```

---

### 🔄 第一轮 (Round 1)
**目标：** 将最大的元素移动到数组末尾。
**比较范围：** 索引 0 到 3 (共 4 次比较)

1.  **比较 `5` 和 `2`**：5 > 2，**交换**
    ```text
    [ 2, 5, 8, 1, 9 ]
    ```
2.  **比较 `5` 和 `8`**：5 < 8，不交换
    ```text
    [ 2, 5, 8, 1, 9 ]
    ```
3.  **比较 `8` 和 `1`**：8 > 1，**交换**
    ```text
    [ 2, 5, 1, 8, 9 ]
    ```
4.  **比较 `8` 和 `9`**：8 < 9，不交换
    ```text
    [ 2, 5, 1, 8, 9 ]
    ```

**🏁 第一轮结束状态：**
```text
[ 2, 5, 1, 8, 9 ]
  └─────┘  └───┘
  未排序    已排序 (9 归位)
```
> 💡 **说明：** 最大的数字 `9` 已经沉底，不需要再参与下一轮比较。

---

### 🔄 第二轮 (Round 2)
**目标：** 将剩余未排序部分中最大的元素移动到倒数第二位。
**比较范围：** 索引 0 到 2 (共 3 次比较)

1.  **比较 `2` 和 `5`**：2 < 5，不交换
    ```text
    [ 2, 5, 1, 8, 9 ]
    ```
2.  **比较 `5` 和 `1`**：5 > 1，**交换**
    ```text
    [ 2, 1, 5, 8, 9 ]
    ```
3.  **比较 `5` 和 `8

## 8.3 Bug 诊断专家

In [65]:
bug_diagnosis_prompt = """你是一位 Python 调试专家。

【错误信息】
```
AttributeError: 'NoneType' object has no attribute 'find_all'
```

【代码】
```python
def parse_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.find_all('div', class_='item')  # 出错行
    return [parse_item(item) for item in items]
```

请分析 3 个最可能的原因及解决方案。"""

response = client.chat([{"role": "user", "content": bug_diagnosis_prompt}])
print(response)



# Python 调试分析：`AttributeError: 'NoneType' object has no attribute 'find_all'`

## 错误原因分析

该错误表明 `soup` 是 `None`，而不是预期的 BeautifulSoup 对象。虽然 `soup = BeautifulSoup(html, 'html.parser')` 通常不会返回 `None`，但在以下几种情况下仍可能发生：

---

## 🔍 最可能的 3 个原因及解决方案

### 1️⃣ `html` 参数为 `None`（最常见）

**原因：** 传入的 `html` 是 `None`，导致 BeautifulSoup 无法正确初始化。

**验证方法：**
```python
print(f"html 类型: {type(html)}, 值: {html}")
```

**解决方案：**
```python
def parse_page(html):
    if not html:
        raise ValueError("HTML 内容不能为空")
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.find_all('div', class_='item')
    return [parse_item(item) for item in items]
```

---

### 2️⃣ BeautifulSoup 未正确导入

**原因：** 如果 `BeautifulSoup` 未正确导入，可能为 `None` 或未定义，导致 `soup` 为 `None`。

**验证方法：**
```python
from bs4 import BeautifulSoup
print(BeautifulSoup)  # 应显示 <class 'bs4.BeautifulSoup'>
```

**解决方案：**
```python
from bs4 import BeautifulSoup  # 确保正确导入

def parse_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    items = 

## 8.4 期末考试出题助手

In [66]:
exam_generator_prompt = """你是一位课程出题专家。

【课程】数据结构与算法
【难度】中等

【知识点】
1. 栈 (LIFO) 的基本概念
2. 队列 (FIFO) 的基本概念
3. 栈和队列的实现

请生成：
- 2 道选择题
- 1 道简答题
- 1 道编程题

附上参考答案。"""

response = client.chat([{"role": "user", "content": exam_generator_prompt}])
print(response)



# 数据结构与算法 - 阶段性测试题

**课程名称**：数据结构与算法  
**难度等级**：中等  
**考核知识点**：栈 (LIFO)、队列 (FIFO)、栈和队列的实现

---

## 一、选择题（每题 5 分，共 10 分）

**1. 若元素进栈的顺序为 1, 2, 3, 4, 5，则下列哪个出栈序列是不可能的？**
A. 5, 4, 3, 2, 1  
B. 4, 5, 3, 2, 1  
C. 3, 2, 5, 4, 1  
D. 3, 4, 1, 2, 5  

**2. 在使用数组实现的循环队列中，为了区分“队空”和“队满”的状态，通常采用的方法是牺牲一个存储单元。假设队列最大容量为 MaxSize，front 指向队头元素，rear 指向队尾元素的下一个位置，则判断队列满的条件是？**
A. `front == rear`  
B. `(rear + 1) % MaxSize == front`  
C. `rear == front + 1`  
D. `front == 0`  

---

## 二、简答题（共 10 分）

**3. 请简述栈（Stack）和队列（Queue）在操作特性上的主要区别，并分别列举一个它们在计算机系统中的典型应用场景。**

---

## 三、编程题（共 20 分）

**4. 请使用两个栈（Stack）来实现一个队列（Queue）。**

**要求：**
1.  定义一个类 `MyQueue`。
2.  实现 `push(x)` 方法：将元素 x 添加到队列尾部。
3.  实现 `pop()` 方法：从队列头部删除并返回元素。
4.  实现 `peek()` 方法：返回队列头部元素但不删除。
5.  实现 `empty()` 方法：判断队列是否为空。
6.  请简要说明你的实现思路（时间复杂度分析更佳）。

*(建议使用 Python 或 C++ 语言作答)*

---

## 参考答案与解析

### 一、选择题

**1. 【答案】D**
*   **解析**：
    *   栈遵循“后进先出”（LIFO）原则。
    *   A 选项：1,2,3,4,5 全部进栈，然后依次出栈，得到 5,4,3,2,1。可行。
    *   B 选项：1,2,3,4 进栈，4 出栈，5 进栈，5 出栈

---

# 总结

## Prompt Engineering 核心技巧

| 技巧 | 适用场景 | 复杂度 |
|------|---------|-------|
| 基础对话 | 简单问答 | ⭐ |
| 角色设定 | 专业任务 | ⭐ |
| Few-Shot | 分类、提取、生成 | ⭐⭐ |
| CoT | 数学、逻辑推理 | ⭐⭐ |
| ReAct | 工具调用、多步骤 | ⭐⭐⭐ |
| Self-Consistency | 重要推理 | ⭐⭐⭐ |
| Tree of Thoughts | 复杂决策 | ⭐⭐⭐ |
| Prompt Chaining | 流水线任务 | ⭐⭐ |
| RAG | 知识库问答 | ⭐⭐⭐ |

## 设计要点

1. **明确角色**：设定 AI 的专业身份
2. **提供背景**：说明任务背景和评分标准
3. **规范输出**：定义清晰的输出格式
4. **示例引导**：提供 Few-shot 示例
5. **约束条件**：明确限制和要求